[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/03_sentiment_im_grossen.ipynb)

# Sitzung 3 — Sentiment im Großen

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI**

Letzte Woche: *eine* Bewertung einordnen. Heute: *hunderte* auf einmal — und die ersten Kennzahlen. Hier fängt „Analytics“ an.

> 💡 Wie immer: alles läuft über den **Mock**, key-frei. Der echte Aufruf kommt nur auf einer kleinen Stichprobe vor (Abschnitt 5).

## 0. Setup — einmal ausführen

In [ ]:
!pip install anthropic --quiet
import json, re, random
from collections import Counter
from datetime import date, timedelta
print('Fertig.')

## 1. Woher kommen hunderte Bewertungen?

Wir **erzeugen** unseren eigenen Datensatz — synthetische Bewertungen zu einem *fiktiven* Produkt (Nimbus Q2). Warum synthetisch? Keine Lizenzprobleme, und wir können das Chaos echter Daten gezielt einbauen (Sarkasmus, Fakes, gemischte Meinungen). Mehr dazu in einer späteren Sitzung.

Führt die Zelle aus — sie definiert den Generator. Details egal; wichtig ist die Funktion **`generate(n)`**.

In [ ]:
"""
Generator fuer synthetische Bewertungen (BDA-Kurs).

Fiktives Produkt: "Nimbus Q2" von der erfundenen Marke "Nimbus Audio".
Voellig fiktiv -> keine echte Marke/Person, keine Lizenzfrage.

Setzt Bewertungen aus variierten Vorlagen zusammen, damit die meisten
EINZIGARTIG sind (realistisch) - mit gezielt eingebautem Daten-Chaos:
  - gemischtes Sentiment, Sarkasmus, Fakes, etwas Englisch, Muell-Zeilen
  - ein paar ABSICHTLICHE exakte Duplikate (fuer die Dedup-Lektion)
  - Wahrheits-Marker (true_sentiment, is_sarcastic, is_fake) fuer spaetere Evaluation
"""
import random
from datetime import date, timedelta

SEED = 42
PRODUCT = "Nimbus Q2"
BRAND = "Nimbus Audio"

# Aspekt-Bausteine
POS = ["der Klang ist hervorragend", "satte Bässe", "die Geräuschunterdrückung ist top",
       "der Akku hält den ganzen Tag", "sitzt super bequem", "Bluetooth verbindet sofort",
       "top verarbeitet", "klasse für den Preis", "die App ist übersichtlich",
       "die Passform ist perfekt", "der Sound ist klar und ausgewogen"]
NEG = ["die App stürzt ständig ab", "der rechte Ohrhörer lädt nicht mehr",
       "die Geräuschunterdrückung rauscht", "viel zu teuer", "die Touch-Steuerung reagiert kaum",
       "das Case wirkt billig", "der Akku ist nach einer Stunde leer",
       "die Verbindung bricht ab", "sie fallen leicht aus dem Ohr", "der Bass ist matschig"]

POS_OPENERS = ["Bin begeistert:", "Wirklich gut:", "Kann ich empfehlen –", "Top Kauf.",
               "Sehr zufrieden:", "Absolute Kaufempfehlung.", "Ich liebe sie:",
               "Klare Sache:", "Rundum gelungen:", "Volle Punktzahl:", "Endlich zufrieden:",
               "Was soll ich sagen –", "Genau richtig:", "Bestellung hat sich gelohnt:"]
NEG_OPENERS = ["Enttäuschend:", "Leider schlecht:", "Finger weg –", "Bin frustriert:",
               "Nicht zu empfehlen.", "Schade um das Geld:", "Ärgerlich:",
               "Reklamiert:", "Bin raus:", "Nie wieder:", "Herbe Enttäuschung:",
               "Das war nichts:", "Zurückgeschickt:", "Vorsicht:"]
NEU_TEMPLATES = ["Ganz okay, {a}, aber nichts Besonderes.",
                 "Erfüllt seinen Zweck. {a_cap}.",
                 "Durchschnittlich. {a_cap}, mehr nicht.",
                 "Habe sie seit Kurzem, {a} – kann noch nicht viel sagen."]
SARCASTIC = ["Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
             "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
             "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
             "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
             "Klasse, nach einer Woche nur noch Rauschen. Wirklich durchdacht.",
             "Perfekt, der linke fällt ständig raus. Genau mein Wunsch.",
             "Herrlich, die Verbindung bricht alle fünf Minuten ab. Danke auch.",
             "Sensationell, das Case bricht beim ersten Öffnen. Qualität eben.",
             "Bravo, nach dem Update ist die Hälfte der Funktionen weg.",
             "Fantastisch leise – weil nach zwei Tagen einfach tot."]
FAKE = ["BESTES PRODUKT EVER!!! Kauft bei www.super-deals-guenstig.example!!!",
        "5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
        "Gutschein Code NIMBUS100 auf meiner Seite jetzt klicken!!!",
        "amazing product best quality buy now discount link in profile",
        "TOP TOP TOP unbedingt kaufen billigster preis hier klicken",
        "gratis versand nur heute!!! rabattcode DEAL22 einlösen!!!",
        "beste kopfhoerer der welt jetzt zuschlagen link im profil",
        "WOW einfach WOW kaufen kaufen kaufen bester preis garantiert",
        "unglaublich guenstig hier klicken und sparen sparen sparen",
        "mega angebot heute -70% nur ueber meinen link!!!"]
ENGLISH = [("Sound quality is great but the app is a disaster.", "mixed"),
           ("Battery life is amazing, best earbuds I have owned.", "positive"),
           ("Stopped working after a week, very disappointed.", "negative"),
           ("Comfortable fit and clear sound, happy with the purchase.", "positive"),
           ("The noise cancelling is weak and the case feels cheap.", "negative")]
JUNK = ["", "   ", ".", "???", "kein kommentar", "-", "n/a", "...", "!!", "??", "keine angabe", "test"]


def _pos(rng):
    o = rng.choice(POS_OPENERS); a = rng.sample(POS, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "positive"

def _neg(rng):
    o = rng.choice(NEG_OPENERS); a = rng.sample(NEG, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "negative"

def _mixed(rng):
    p = rng.choice(POS); n = rng.choice(NEG)
    conn = rng.choice([" - aber ", ", allerdings ", ". Leider ", ", jedoch "])
    return f"{p[0].upper()+p[1:]}{conn}{n}.", "mixed"

def _neutral(rng):
    a = rng.choice(POS + NEG)
    t = rng.choice(NEU_TEMPLATES).format(a=a, a_cap=a[0].upper()+a[1:])
    return t, "neutral"

def _rating_for(truth, rng):
    return {"positive": [4,5,5], "negative": [1,1,2], "mixed": [2,3,4],
            "neutral": [3,3,4], "fake": [5,5], }.get(truth, [1,3,5]) and \
           rng.choice({"positive":[4,5,5],"negative":[1,1,2],"mixed":[2,3,4],
                       "neutral":[3,3,4],"fake":[5,5]}.get(truth,[1,3,5]))

def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:   text, truth = _pos(rng)
        elif r < 0.60: text, truth = _neg(rng)
        elif r < 0.72: text, truth = _mixed(rng)
        elif r < 0.80: text, truth = _neutral(rng)
        elif r < 0.88: text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.93: text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98: text, truth = rng.choice(ENGLISH)
        else:          text, truth = rng.choice(JUNK), "junk"
        day = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day)).isoformat(),
            "product": PRODUCT,
            "rating": _rating_for(truth, rng),
            "text": text,
            "true_sentiment": truth,
            "is_sarcastic": text in SARCASTIC,
            "is_fake": text in FAKE,
        })
    # genau 3 absichtliche exakte Duplikate für die Dedup-Lektion
    if n > 20:
        for j, src in enumerate([5, 12, 30]):
            rows.append(dict(rows[src], review_id=f"R{n+j:04d}"))
    rng.shuffle(rows)
    return rows


def save(rows, csv_path, hide_truth=False):
    import csv
    fields = ["review_id","date","product","rating","text"]
    if not hide_truth: fields += ["true_sentiment","is_sarcastic","is_fake"]
    with open(csv_path,"w",newline="",encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader(); w.writerows(rows)

Und die `classify`-Funktion von letzter Woche (Mock + echt):

In [ ]:
# Minimale, eigenständige classify-Funktion (Mock + echt).
# Bewusst einfach - wir brauchen nur:
# "eine Bewertung rein, Sentiment+Begründung raus".
import json, re

SENTIMENTS = ["positive", "negative", "neutral", "mixed"]

def _mock_classify(text):
    """Attrappe: plausibel, deterministisch. Liest Sarkasmus wörtlich (Lehrzweck)."""
    t = (text or "").lower()
    pos = sum(w in t for w in ["gut","top","super","toll","hervorragend","bequem",
                                "stabil","klasse","liebe","genial","perfekt","great","love"])
    neg = sum(w in t for w in ["schlecht","kaputt","teuer","stürzt","rauscht","billig",
                                "nervt","enttäuscht","langsam","mangel","bad","broken"])
    if pos and neg: s = "mixed"
    elif pos: s = "positive"
    elif neg: s = "negative"
    else: s = "neutral"
    return {"sentiment": s, "reason": f"Mock: {pos} positive-, {neg} negative-Signale erkannt."}

def _real_classify(text, client, model="claude-sonnet-5"):
    prompt = (f'Ordne die folgende Produktbewertung ein. Antworte NUR mit JSON:\n'
              f'{{"sentiment": "positive|negative|neutral|mixed", "reason": "kurze Begruendung"}}\n\n'
              f'Bewertung: """{text}"""')
    resp = client.messages.create(model=model, max_tokens=200,
                                  messages=[{"role":"user","content":prompt}])
    raw = next((b.text for b in resp.content if hasattr(b,"text")), "")
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    return json.loads(m.group(0)) if m else {"sentiment":"neutral","reason":"parse error"}

def classify(text, use_real=False, client=None):
    """Die eine Funktion, die ihr aufruft. Standard = Mock; echt mit client."""
    return _real_classify(text, client) if use_real else _mock_classify(text)

SAMPLE_REVIEWS = [
    "Der Klang ist wirklich hervorragend, aber die App stürzt ständig ab.",
    "Absolut top verarbeitet und bequem. Klare Kaufempfehlung!",
    "Nach zwei Wochen kaputt. Nie wieder.",
    "Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",  # sarcasm
    "Ganz okay, erfüllt seinen Zweck.",
]

Jetzt erzeugen wir 300 Bewertungen und schauen in die ersten hinein:

In [ ]:
reviews = generate(300)
print(f'{len(reviews)} Bewertungen erzeugt.\n')
for r in reviews[:5]:
    print(f"[{r['rating']}*] {r['text'][:60]}")

## 2. Von einer zu hunderten: die Schleife

In Sitzung 2 haben wir *eine* Bewertung eingeordnet. Um *alle* einzuordnen, wiederholen wir das in einer **Schleife** — für jede Bewertung ein Aufruf.

> 💡 Beim Mock geht das in einer Sekunde. Beim *echten* LLM wäre das hunderte Aufrufe — langsam und kostenpflichtig. Merkt euch das (Thema in Sitzung 11).

In [ ]:
analysiert = []
for r in reviews:
    ergebnis = classify(r['text'])          # Mock
    analysiert.append({**r, **ergebnis})
print('Analysiert:', len(analysiert))
print('Beispiel  :', analysiert[0]['sentiment'], '|', analysiert[0]['text'][:50])

## 3. Aggregieren: das Gesamtbild

Eine einzelne Bewertung ist eine Anekdote. **Hunderte zusammen** sind ein Signal. Zählen wir, wie oft jedes Sentiment vorkommt:

In [ ]:
counts = Counter(a['sentiment'] for a in analysiert)
print(counts)

n = len(analysiert)
print('\nVerteilung:')
for s, c in counts.most_common():
    print(f'  {s:9} {c:4}  ({round(100*c/n)}%)')

Das ist eure erste **Kennzahl**: der Anteil positiver / negativer Bewertungen über den ganzen Datensatz. Genau das will eine Produktmanagerin auf einen Blick.

## 4. Eure Aufgabe: vergleichen

Eine Kennzahl über *alles* ist nett. Interessanter wird es beim **Vergleichen**. Nehmt eine **Teilmenge** (z. B. nur schlecht bewertete) und schaut, wie sich das Sentiment unterscheidet.

In [ ]:
# Teilmenge: nur Bewertungen mit 1 oder 2 Sternen
schlecht = [a for a in analysiert if a['rating'] <= 2]
print(f'{len(schlecht)} Bewertungen mit <=2 Sternen')
print(Counter(a['sentiment'] for a in schlecht))

> ✏️ **Eure Aufgabe:** Ändert die Bedingung. Vergleicht z. B. `rating >= 4` (gute Sterne) mit dem Gesamtbild. Stimmen Sterne und LLM-Sentiment überein? Wo weichen sie ab — und warum könnte das sein?

## Der tiefere Punkt: braucht man *alle* Daten?

Wir haben 300 Bewertungen eingeordnet. Aber: braucht man wirklich *alle*, um das Gesamtbild zu kennen? Oft reicht eine **Stichprobe** — eine zufällige Auswahl, die den Datensatz repräsentiert. Das spart Zeit und (beim echten LLM) Geld.

In [ ]:
stichprobe = random.sample(analysiert, 50)   # 50 zufaellig gezogen
c_gesamt = Counter(a['sentiment'] for a in analysiert)
c_probe  = Counter(a['sentiment'] for a in stichprobe)
print('Gesamt (300):', {k: round(100*v/300) for k,v in c_gesamt.items()})
print('Probe  (50) :', {k: round(100*v/50)  for k,v in c_probe.items()})

> 💡 **Kernpunkt:** Die Stichprobe kommt dem Gesamtbild oft nah — aber nicht exakt. Wie groß eine Stichprobe sein muss, um verlässlich zu sein, ist eine echte statistische Frage (klassische Big-Data-Analytics!). Für jetzt: weniger Daten können reichen, wenn sie *repräsentativ* sind.

## 5. Kurz echt: eine kleine Stichprobe mit dem echten LLM

Alles bisher lief über den Mock. Zum Abschluss: dieselbe Schleife, aber echt — und **nur auf 5 Bewertungen** (echte Aufrufe kosten Zeit und Geld).

> ℹ️ Nur mit Schlüssel (Secrets-Panel, `ANTHROPIC_API_KEY`). Ohne Schlüssel überspringen — der Mock reicht.

In [ ]:
# Nur mit Schlüssel:
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

for r in reviews[:5]:
    e = classify(r['text'], use_real=True, client=client)
    print(f"{e['sentiment']:9} | {r['text'][:55]}")

> 🎓 **Vergleich:** Wie ordnet das echte LLM diese 5 ein — verglichen mit dem Mock? Achtet besonders auf die sarkastischen und gemischten Fälle.

## 6. Geschafft — und Ausblick

Ihr habt hunderte Bewertungen eingeordnet und zu Kennzahlen verdichtet — das ist der Kern von *Analytics*.

**Nächste Woche (28.10):** die Daten sind messy (fehlende Felder, Duplikate, Fremdsprachen). Wir bringen sie in Form — bevor wir in Sitzung 5 anfangen, *Themen* zu extrahieren.

> 💡 Von *einer* Zahl (%) zu *vielen* Einsichten: das bauen wir Schritt für Schritt aus.